In [1]:
from dotenv import load_dotenv
load_dotenv()

import sys
sys.path.append("..")

In [2]:
from ingest import load_faq_data
documents = load_faq_data()

In [3]:
documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

len(documents_llm)

113

In [4]:
documents = documents_llm

doc = documents[0]
print(doc["id"])
print(doc["question"])
print(doc["answer"])

74eb249bbf
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


In [5]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [6]:
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [7]:
from openai import OpenAI

openai_client = OpenAI()

In [8]:
import json

user_prompt = json.dumps(doc)

In [9]:
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

In [10]:
response = openai_client.responses.parse(
    model="gpt-5.4-mini",
    input=messages,
    text_format=Questions
)

In [11]:
result = response.output_parsed

print(result)

questions=['I just found this course, is it still okay to join now?', 'Am I too late to start the course and participate?', 'Can I enroll after the course has already started?', 'If I join late, can I still get a certificate somehow?', 'What do I need to do to be eligible for the certificate if I’m joining now?']


In [12]:
print(result.questions)

['I just found this course, is it still okay to join now?', 'Am I too late to start the course and participate?', 'Can I enroll after the course has already started?', 'If I join late, can I still get a certificate somehow?', 'What do I need to do to be eligible for the certificate if I’m joining now?']


In [13]:
from evaluation_utils import llm_structured

In [14]:
result, usage = llm_structured(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

print(result.questions)

['Can I still join the course if I just found out about it?', 'Am I allowed to start the course late and still participate?', 'If I join after the course has already started, can I still get a certificate?', 'What do I need to do in order to receive the certificate if I’m joining now?', 'Is it too late to enroll, or can I still come in and take part?']


In [15]:
usage.input_tokens, usage.output_tokens

(207, 94)

In [16]:
from evaluation_utils import calc_price

In [17]:
cost = calc_price(usage)

cost

{'input_cost': 0.00015525, 'output_cost': 0.000423, 'total_cost': 0.00057825}

In [18]:
records = []

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["id"]
    })

records

[{'question': 'Can I still join the course if I just found out about it?',
  'document': '74eb249bbf'},
 {'question': 'Am I allowed to start the course late and still participate?',
  'document': '74eb249bbf'},
 {'question': 'If I join after the course has already started, can I still get a certificate?',
  'document': '74eb249bbf'},
 {'question': 'What do I need to do in order to receive the certificate if I’m joining now?',
  'document': '74eb249bbf'},
 {'question': 'Is it too late to enroll, or can I still come in and take part?',
  'document': '74eb249bbf'}]

## Batch

In [19]:
from evaluation_utils import llm_structured_retry

In [20]:
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["id"]
        })

    return results, usage

In [21]:
from tqdm.auto import tqdm

ground_truth = []
usages = []

for doc in tqdm(documents[:5]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

  0%|          | 0/5 [00:00<?, ?it/s]

In [22]:
ground_truth

[{'question': 'I found this course late — can I still join, or is it too late now?',
  'document': '74eb249bbf'},
 {'question': 'If I start the course after it already began, am I still able to get a certificate?',
  'document': '74eb249bbf'},
 {'question': 'Do I need to join at the beginning, or can I enter the course in the middle and still participate?',
  'document': '74eb249bbf'},
 {'question': 'Is there any deadline I should know about if I want a certificate after joining late?',
  'document': '74eb249bbf'},
 {'question': 'Can I still take part in the course now, and what do I need to do to be eligible for the certificate?',
  'document': '74eb249bbf'},
 {'question': 'Do I actually need a confirmation email after signing up for LLM Zoomcamp, or am I already good to go?',
  'document': '977bf7786c'},
 {'question': 'If I registered for the course but never got any email, can I still start the lessons and homework?',
  'document': '977bf7786c'},
 {'question': 'Is there some approve

In [23]:
usages

[ResponseUsage(input_tokens=207, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=116, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=323),
 ResponseUsage(input_tokens=238, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=129, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=367),
 ResponseUsage(input_tokens=315, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=119, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=434),
 ResponseUsage(input_tokens=379, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=116, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=495),
 ResponseUsage(input_tokens=381, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=110, output_token

### Parallel processing

In [24]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [25]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, documents, generate_ground_truth)

  0%|          | 0/113 [00:00<?, ?it/s]

In [26]:
ground_truth = []
usages = []

for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)

len(ground_truth)

565

In [27]:
from evaluation_utils import calc_price

total_cost = 0.0

for usage in usages:
    cost = calc_price(usage)
    total_cost = total_cost + cost["total_cost"]

total_cost

0.08765250000000005

In [28]:
from evaluation_utils import calc_total_price

calc_total_price(usages)

0.08765250000000005

In [29]:
import pandas as pd

df_ground_truth = pd.DataFrame(ground_truth)

In [30]:
df_ground_truth.to_csv("data/ground_truth-new.csv", index=False)